<a href="https://colab.research.google.com/github/sahferr/ciencia-da-computacao/blob/main/treino_prova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

treino_dl, val_dl = criar_loaders(LOCAL / 'train')
modelo = UNet().to(DEVICE)
historico = treinar(modelo, treino_dl, val_dl, EPOCAS)
historico.to_csv(PASTA_DRIVE / 'historico_treino.csv', index=False)
print(f'Modelo salvo em {ARQUIVO_MODELO}')

Treino: 229 | Validação: 40
Época 01 | treino 0.1206 | val 0.1183
Época 02 | treino 0.1000 | val 0.1001
Época 03 | treino 0.0862 | val 0.0772
Época 04 | treino 0.0783 | val 0.0765
Época 05 | treino 0.0703 | val 0.0684
Época 06 | treino 0.0641 | val 0.0672
Época 07 | treino 0.0596 | val 0.0564
Época 08 | treino 0.0560 | val 0.0563
Época 09 | treino 0.0548 | val 0.0536
Época 10 | treino 0.0531 | val 0.0506
Época 11 | treino 0.0509 | val 0.0508
Época 12 | treino 0.0511 | val 0.0506
Época 13 | treino 0.0495 | val 0.0468
Época 14 | treino 0.0488 | val 0.0490
Época 15 | treino 0.0480 | val 0.0494
Época 16 | treino 0.0486 | val 0.0502
Época 17 | treino 0.0486 | val 0.0457
Época 18 | treino 0.0478 | val 0.0472
Época 19 | treino 0.0472 | val 0.0468
Época 20 | treino 0.0463 | val 0.0459
Época 21 | treino 0.0469 | val 0.0499
Época 22 | treino 0.0465 | val 0.0469
Época 23 | treino 0.0460 | val 0.0458
Época 24 | treino 0.0454 | val 0.0466
Época 25 | treino 0.0450 | val 0.0457
Época 26 | treino 0.04

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
Path('/content/drive/MyDrive/endotelio').mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [2]:
import sys, importlib
sys.path.append('/content/drive/MyDrive/endotelio')
import funcoes
importlib.reload(funcoes)
from funcoes import *

In [3]:
import shutil
from torch.utils.data import Dataset, DataLoader, random_split

LOCAL = Path('dataset')
VAL_SIZE = 0.15
EPOCAS = 50
BATCH = 8

if not LOCAL.exists():
    shutil.copytree(DATASET, LOCAL)
print(f"Treino: {len(listar_conjunto(LOCAL / 'train'))} imagens")

Treino: 269 imagens


In [4]:
class CelulasDataset(Dataset):
    def __init__(self, pasta: Path):
        self.pares = listar_conjunto(pasta)

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, i):
        img_path, txt_path = self.pares[i]
        img = carregar_cinza(img_path)
        centros = ler_centros(txt_path)
        entrada = corrigir_iluminacao(img).astype(np.float32) / 255
        mapa = gerar_mapa(img.shape, centros)
        mascara = gerar_mascara(img.shape, centros).astype(np.float32)
        return tuple(torch.from_numpy(ajustar_tamanho(a)[None]) for a in (entrada, mapa, mascara))

In [5]:
def perda_mascarada(pred, alvo, mascara, peso_centro: float = 10):
    peso = mascara * (1 + peso_centro * alvo)
    return (peso * (pred - alvo) ** 2).sum() / peso.sum().clamp_min(1)

In [6]:
def aumentar(*tensores):
    if torch.rand(1) < 0.5:
        tensores = [t.flip(-1) for t in tensores]
    if torch.rand(1) < 0.5:
        tensores = [t.flip(-2) for t in tensores]
    return tensores

In [7]:
def rodar_epoca(modelo, loader, otimizador=None) -> float:
    treinando = otimizador is not None
    modelo.train(treinando)
    total = 0
    with torch.set_grad_enabled(treinando):
        for entrada, alvo, mascara in loader:
            entrada, alvo, mascara = (t.to(DEVICE) for t in (entrada, alvo, mascara))
            if treinando:
                entrada, alvo, mascara = aumentar(entrada, alvo, mascara)
            perda = perda_mascarada(modelo(entrada), alvo, mascara)
            if treinando:
                otimizador.zero_grad()
                perda.backward()
                otimizador.step()
            total += perda.item()
    return total / len(loader)

In [8]:
def criar_loaders(pasta: Path) -> tuple[DataLoader, DataLoader]:
    completo = CelulasDataset(pasta)
    n_val = int(VAL_SIZE * len(completo))
    treino_ds, val_ds = random_split(
        completo, [len(completo) - n_val, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f'Treino: {len(treino_ds)} | Validação: {len(val_ds)}')
    return (DataLoader(treino_ds, batch_size=BATCH, shuffle=True, num_workers=2),
            DataLoader(val_ds, batch_size=BATCH))

In [9]:
def treinar(modelo, treino_dl, val_dl, epocas: int, lr: float = 1e-3) -> pd.DataFrame:
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    historico, melhor = [], float('inf')
    for epoca in range(1, epocas + 1):
        p_treino = rodar_epoca(modelo, treino_dl, otimizador)
        p_val = rodar_epoca(modelo, val_dl)
        historico.append({'epoca': epoca, 'treino': p_treino, 'val': p_val})
        if p_val < melhor:
            melhor = p_val
            torch.save(modelo.state_dict(), ARQUIVO_MODELO)
        print(f'Época {epoca:02d} | treino {p_treino:.4f} | val {p_val:.4f}')
    return pd.DataFrame(historico)

In [11]:
treino_dl, val_dl = criar_loaders(LOCAL / 'train')
modelo = UNet().to(DEVICE)
historico = treinar(modelo, treino_dl, val_dl, EPOCAS)
historico.to_csv(PASTA_DRIVE / 'historico_treino.csv', index=False)
print(f'Modelo salvo em {ARQUIVO_MODELO}')

Treino: 229 | Validação: 40
Época 01 | treino 0.1206 | val 0.1183
Época 02 | treino 0.1000 | val 0.1001
Época 03 | treino 0.0862 | val 0.0772
Época 04 | treino 0.0783 | val 0.0765
Época 05 | treino 0.0703 | val 0.0684
Época 06 | treino 0.0641 | val 0.0672
Época 07 | treino 0.0596 | val 0.0564
Época 08 | treino 0.0560 | val 0.0563
Época 09 | treino 0.0548 | val 0.0536
Época 10 | treino 0.0531 | val 0.0506
Época 11 | treino 0.0509 | val 0.0508
Época 12 | treino 0.0511 | val 0.0506
Época 13 | treino 0.0495 | val 0.0468
Época 14 | treino 0.0488 | val 0.0490
Época 15 | treino 0.0480 | val 0.0494
Época 16 | treino 0.0486 | val 0.0502
Época 17 | treino 0.0486 | val 0.0457
Época 18 | treino 0.0478 | val 0.0472
Época 19 | treino 0.0472 | val 0.0468
Época 20 | treino 0.0463 | val 0.0459
Época 21 | treino 0.0469 | val 0.0499
Época 22 | treino 0.0465 | val 0.0469
Época 23 | treino 0.0460 | val 0.0458
Época 24 | treino 0.0454 | val 0.0466
Época 25 | treino 0.0450 | val 0.0457
Época 26 | treino 0.04